In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import pickle as pkl


In [2]:
df = pd.read_csv(r"C:\Hassan\AI Projects and learning 2026\Projects\Loan Approval\data\loan_data.csv")

In [3]:
df = df.drop(columns=["person_age","person_emp_exp","loan_amnt","person_gender","person_education","cb_person_cred_hist_length","credit_score"])
# Combine EDUCATION, PERSONAL, MEDICAL into EPM
df['loan_intent'] = df['loan_intent'].map({
    'EDUCATION': 'EPM',
    'PERSONAL': 'EPM', 
    'MEDICAL': 'EPM',
    'nan': 'EPM',
    'VENTURE': 'VENTURE',    # keep as is
    'HOMEIMPROVEMENT': 'HOMEIMPROVEMENT'  # keep as is
})
df["loan_intent"] = df['loan_intent'].fillna("EPM")
df['loan_intent'].unique()

<StringArray>
['EPM', 'VENTURE', 'HOMEIMPROVEMENT']
Length: 3, dtype: str

In [4]:
df.head(1)

,person_income,person_home_ownership,loan_intent,loan_int_rate,loan_percent_income,previous_loan_defaults_on_file,loan_status
0,71948.0,RENT,EPM,16.02,0.49,No,1


In [5]:

# Encode
categorical_cols = ["person_home_ownership","loan_intent","previous_loan_defaults_on_file"]
encoder = OneHotEncoder(sparse_output=False)

    
data_encoded = encoder.fit_transform(df[categorical_cols])
df = df.drop(categorical_cols, axis=1)
df = pd.concat([df, pd.DataFrame(data_encoded, columns=encoder.get_feature_names_out(), index=df.index)], axis=1)


In [6]:
df.head()

,person_income,loan_int_rate,loan_percent_income,loan_status,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EPM,loan_intent_HOMEIMPROVEMENT,loan_intent_VENTURE,previous_loan_defaults_on_file_No,previous_loan_defaults_on_file_Yes
0,71948.0,16.02,0.49,1,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
1,12282.0,11.14,0.08,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
2,12438.0,12.87,0.44,1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,79753.0,15.23,0.44,1,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4,66135.0,14.27,0.53,1,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


In [7]:
X = df.drop(columns ="loan_status")
y = df["loan_status"]

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

In [9]:
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

In [10]:
from sklearn.metrics import classification_report, confusion_matrix

In [11]:
xgb_model = XGBClassifier()

In [12]:
hgb_model = HistGradientBoostingClassifier()

In [13]:
scaler = MinMaxScaler()

In [14]:
scaler.fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [15]:
xgb_model.fit(X_train_scaled,y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [16]:
hgb_model.fit(X_train_scaled, y_train)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dt

In [17]:
y_xgb = xgb_model.predict(X_test_scaled)
y_hgb = hgb_model.predict(X_test_scaled)

In [18]:
print("XGB MODEL:")
print(classification_report(y_test, y_xgb))
print("Hist Gradient MODEL:")
print(classification_report(y_test, y_hgb))

XGB MODEL:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      3593
           1       0.87      0.79      0.83       907

    accuracy                           0.93      4500
   macro avg       0.91      0.88      0.89      4500
weighted avg       0.93      0.93      0.93      4500

Hist Gradient MODEL:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      3593
           1       0.88      0.77      0.82       907

    accuracy                           0.93      4500
   macro avg       0.91      0.87      0.89      4500
weighted avg       0.93      0.93      0.93      4500



In [19]:
importances = xgb_model.feature_importances_
feat_names = X_train_scaled.columns

# Create a DataFrame for easy sorting and plotting
df_importance = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
df_importance = df_importance.sort_values(by='Importance', ascending=True)

In [20]:
df_importance

,Feature,Importance
11,previous_loan_defaults_on_file_Yes,0.000000
3,person_home_ownership_MORTGAGE,0.002008
4,person_home_ownership_OTHER,0.002855
7,loan_intent_EPM,0.003717
0,person_income,0.006859
1,loan_int_rate,0.009030
8,loan_intent_HOMEIMPROVEMENT,0.009444
9,loan_intent_VENTURE,0.010223
2,loan_percent_income,0.012401
5,person_home_ownership_OWN,0.016960


In [30]:
useless_features = []

for i in range(df_importance.shape[0]):  # [0] = rows
    if df_importance.iloc[i]['Importance'] < 0.005:
        useless_features.append(df_importance.iloc[i]['Feature'])

print(f"Useless features ({len(useless_features)}):")
print(useless_features)

Useless features (4):
['previous_loan_defaults_on_file_Yes', 'person_home_ownership_MORTGAGE', 'person_home_ownership_OTHER', 'loan_intent_EPM']


In [22]:
print(confusion_matrix(y_test, y_xgb))
print(confusion_matrix(y_test, y_hgb))

[[3490  103]
 [ 191  716]]
[[3495   98]
 [ 212  695]]


In [31]:
X_train_scaled.drop(columns=useless_features).head()
X_test_scaled.drop(columns=useless_features).head()

,person_income,loan_int_rate,loan_percent_income,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_HOMEIMPROVEMENT,loan_intent_VENTURE,previous_loan_defaults_on_file_No
0,0.010553,0.462277,0.196970,0.0,1.0,0.0,0.0,1.0
1,0.009195,0.416324,0.348485,0.0,1.0,0.0,0.0,0.0
2,0.004232,0.553498,0.121212,0.0,1.0,0.0,0.0,0.0
3,0.010575,0.628944,0.454545,0.0,1.0,1.0,0.0,1.0
4,0.014837,0.759259,0.075758,0.0,1.0,0.0,0.0,1.0


In [33]:
pkl.dump(xgb_model, open(r'C:\Hassan\AI Projects and learning 2026\Projects\Loan Approval\model.pkl', 'wb'))
pkl.dump(scaler, open(r'C:\Hassan\AI Projects and learning 2026\Projects\Loan Approval\scaler.pkl', 'wb'))
pkl.dump(encoder, open(r'C:\Hassan\AI Projects and learning 2026\Projects\Loan Approval\encoder.pkl', 'wb'))